# <center> <img src="figs/LogoUFSCar.jpg" alt="Logo UFScar" width="110" align="left"/>  <br/> <center>Universidade Federal de São Carlos (UFSCar)<br/><font size="4"> Departamento de Computação, campus Sorocaba</center></font>
</p>

<font size="4"><center><b>Disciplina: Aprendizado de Máquina</b></center></font>
  
<font size="3"><center>Prof. Dr. Tiago A. Almeida</center></font>

## <center>Projeto Final</center>

**Nome**: Vinícius Henrique de Proença Cavalcanti

**RA**: 839901


In [1]:
from scripts.preprocessamento import preprocess, remove_outliers, clean_column_names
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from scripts.analise_resultados import evaluate
from sklearn.impute import IterativeImputer
from scripts.experimentos import ModelTrain
from sklearn.naive_bayes import GaussianNB
from matplotlib import pyplot as plt
from sklearn.svm import SVC
import missingno as msno
import seaborn as sns
import pandas as pd
import numpy as np

---
### Análise exploratória

Nesta seção, deve ser feita a leitura da base de dados e todas as análises necessárias para interpretar e analisar os dados, tais como:
* Significado de cada atributo
* Medidas descritivas
* Gráficos

In [2]:
rhp_df = pd.read_csv('data/rhp_data.csv')
test_df = pd.read_csv('data/test.csv')
train_df = pd.read_csv('data/train.csv')

In [ ]:
test_df.head()

In [ ]:
train_df.head()

In [5]:
rhp_train_df = train_df.merge(rhp_df, on='Id', how='left')

In [6]:
rhp_test_df = test_df.merge(rhp_df, on='Id', how='left')

In [ ]:
rhp_train_df.head()

In [ ]:
rhp_train_df.shape

In [ ]:
rhp_train_df.dtypes

apesar de idade e fc serem númericos estão como object. os valores devem ser convertidos para o tipo certo para serem analisados

In [10]:
rhp_train_df['IDADE'] = pd.to_numeric(rhp_train_df['IDADE'], errors='coerce')
rhp_train_df['FC'] = pd.to_numeric(rhp_train_df['FC'], errors='coerce')

In [11]:
rhp_test_df['IDADE'] = pd.to_numeric(rhp_test_df['IDADE'], errors='coerce')
rhp_test_df['FC'] = pd.to_numeric(rhp_test_df['FC'], errors='coerce')

De inicio, alguns atributos parecem não causar efeito na identificação de patologias cárdiacas e por isso serão removidos. os atributos são:
- id
- atendimento
- dn
- convenio
- altura

Valores categoricos que serão utilizados na análise devem ser convertidos para númericos. apesar de alguns modelos de classificação aceitarem valores categóricos a maioria não aceita e os modelos lidam melhor com dados númericos.

Alguns atributos possuem valores minimos não condizentes, como:
- peso negativo
- idade negativa e zerada
- indice de massa corporal zero

outros atributos possuem valores minimos abaixo do que se espera, porem em casos criticos os valores realmente podem chegar a este extremo, então essse atributos podem ser considerados como casos fora da curva

em relação aos valores maximos alguns atributos possuem valor muito acima do que se espera, categorizando como outliers, esses são:
- PA sistolica acima de 200
- idade acima da idade de pediatria
- IMC acima de 100
- fc acima de 300

In [ ]:
rhp_train_df.describe()

In [ ]:
fig, ax = plt.subplots(4, 2, figsize=(15, 10))

sns.boxplot(x='Peso', data=rhp_train_df, ax=ax[0, 0])
sns.boxplot(x='IDADE', data=rhp_train_df, ax=ax[0, 1])
sns.boxplot(x='IMC', data=rhp_train_df, ax=ax[1, 0])
sns.boxplot(x='PA SISTOLICA', data=rhp_train_df, ax=ax[1, 1])
sns.boxplot(x='PA DIASTOLICA', data=rhp_train_df, ax=ax[2, 0])
sns.boxplot(x='FC', data=rhp_train_df, ax=ax[2, 1])
sns.boxplot(x='Altura', data=rhp_train_df, ax=ax[3, 0])

plt.tight_layout()
plt.show()

todas entradas que possuem peso menor ou igual a zero tambem não possuem o imc, que seria um indicador para preencher os pesos faltantes

In [ ]:
rhp_train_df[rhp_train_df['Peso'] <= 0].head()

In [ ]:
rhp_train_df[rhp_train_df['Peso'] <= 0].shape

In [ ]:
(rhp_train_df[rhp_train_df['Peso'] <= 0]['IMC'] > 0).value_counts()

para idade, imc, pa sistolica e fc temos valores negativos, zerados e acima do esperado, entretanto, somados, essas entradas não equivalem a 10% do conjunto de treino

In [ ]:
rhp_train_df[(rhp_train_df['IDADE'] <= 0) | (rhp_train_df['IDADE'] >= 22)].head()

In [ ]:
rhp_train_df[(rhp_train_df['IDADE'] <= 0) | (rhp_train_df['IDADE'] >= 22)].shape

In [ ]:
rhp_train_df[rhp_train_df['IMC'] > 100].head()

In [ ]:
rhp_train_df[rhp_train_df['IMC'] > 100].shape

In [ ]:
rhp_train_df[rhp_train_df['PA SISTOLICA'] > 200].head()

In [ ]:
rhp_train_df[rhp_train_df['FC'] > 480].head()

In [ ]:
rhp_train_df[rhp_train_df['FC'] > 480].shape

alem dos valores fora dos niveis esperados existem tambem valores faltantes.

todos as observações com 'peso' vazio tambem não possuem 'imc'. pode haver uma correlação entre pacientes acima do peso que não tiveram o peso coletado, ou então, que o medico responsável deixou de pesar algum paciente por alguma outra razão que não está na base de dados. frequencia cardiaca (sistolica e diastolica) pode tambem não ter sido aferida e a razão do vazio é simplesmente a não coleta do dado. essas são algumas possibilidades que não temos como confirmar sem o auxilio de quem coletou os dados, e a introdução de informações sem a identificação desses padrões pode levar a inserção de viés dentro da base



pa sistolica e diastolica possuem aprox 40% dos valores nulos, o que torna possivel o preenchimento com base nos valores não nulos e outros atributos tambem, entretanto, valores de pressão arterial variam muito e dependem de muitos fatores que não estão presentes no conjunto como: repouso, estresse, colesterol, falta de atividade fisica... ppa é um valor que foi criado em cima dos valores de pressao sistolica e diastolica e possui uma porcentagem menor de valores vazios

a maioria das entradas vazias de imc possuem peso, entretanto, não possuem altura, que é um dos valores do calculo de imc

a idade depende de fatores como genetica, saude, nutrição e outros, que são valores não presentes no conjunto. a idade pode ser estimada a partir da altura e genero mas são aproximações que podem não ser fidedignas

para decidir qual a melhor forma de imputar os valores continuos serão analisados três metodos diferentes de imputação de dados

In [ ]:
rhp_train_df.isna().sum()

In [ ]:
msno.matrix(rhp_train_df)

In [26]:
rhp_train_num_df = rhp_train_df.select_dtypes(include=[np.number]).drop(columns=['Id', 'PA SISTOLICA', 'PA DIASTOLICA'])

In [27]:
rhp_train_num_df = clean_column_names(rhp_train_num_df)
rhp_train_num_df = remove_outliers(rhp_train_num_df)
rhp_train_num_df = rhp_train_num_df.dropna()

In [ ]:
missing_mask = np.random.rand(*rhp_train_num_df.shape) < 0.4
simulated_data = rhp_train_num_df.mask(missing_mask)
simulated_data.head()

In [29]:
original_mean = simulated_data.mean()
original_std = simulated_data.std()

In [30]:
original_values = (rhp_train_num_df - rhp_train_num_df.mean()) / rhp_train_num_df.std()
simulated_data = (simulated_data - simulated_data.mean()) / simulated_data.std()

In [ ]:
median_imputer = SimpleImputer(strategy="median")
median_imputed_data = pd.DataFrame(
    median_imputer.fit_transform(simulated_data), columns=simulated_data.columns
)
median_mae = mean_absolute_error(original_values, median_imputed_data)
median_mse = root_mean_squared_error(original_values, median_imputed_data)
r2 = r2_score(original_values, median_imputed_data)
print(f"Median Imputer MAE: {median_mae:.2f}")
print(f"Median Imputer MSE: {median_mse:.2f}")
print(f"Median Imputer R2: {r2:.2f}")

In [ ]:
median_imputed_data.head()

In [ ]:
knn_imputer = KNNImputer(n_neighbors=5)
knn_imputed_data = pd.DataFrame(
    knn_imputer.fit_transform(simulated_data), columns=simulated_data.columns
)
knn_mae = mean_absolute_error(original_values, knn_imputed_data)
knn_mse = root_mean_squared_error(original_values, knn_imputed_data)
r2 = r2_score(original_values, knn_imputed_data)
print(f"KNN Imputer MAE: {knn_mae:.2f}")
print(f"KNN Imputer RMSE: {knn_mse:.2f}")
print(f"KNN Imputer R2: {r2:.2f}")

In [ ]:
knn_imputed_data.head()

In [ ]:
multiple_imputer = IterativeImputer(max_iter=200, initial_strategy="median")
multiple_imputed_data = pd.DataFrame(
    multiple_imputer.fit_transform(simulated_data), columns=simulated_data.columns
)
multiple_mae = mean_absolute_error(original_values, multiple_imputed_data)
multiple_mse = root_mean_squared_error(original_values, multiple_imputed_data)
r2 = r2_score(original_values, multiple_imputed_data)
print(f"Multiple Imputer MAE: {multiple_mae:.2f}")
print(f"Multiple Imputer RMSE: {multiple_mse:.2f}")
print(f"Multiple Imputer R2: {r2:.2f}")

In [36]:
multiple_imputed_data = (multiple_imputed_data * original_std) + original_mean

In [ ]:
multiple_imputed_data.head()

hda 2 possui mais de 90% dos dados nulos inviabilizando o preechimento.  
os valores categoricos serão padronizados, agrupados e convertidos para númerico, utilizando a técnica de one-hot-enconding, para garantir que os modelos não tratem as categorias como ordinais, isto é, que possuem uma hierarquia ou que uma se sobrepoem sobre a outra.

sexo:
- Masculino
- Feminino
- Indefinido

hda1:
- assintomatico
- sintomas_cardiacos
- sintomas_gerais

pulsos:
- normais
- alterados

b2:
- normal
- alterado

sopro:
- sistolico
- continuo
- diastolico

motivo1:
- condicao_cardiaca
- checkup
- outro

ppa:
- normal
- nao_calculado
- pre_hipertensao
- hipertensao
- outro

In [ ]:
rhp_train_df['SEXO'].value_counts()

In [ ]:
rhp_train_df['HDA 1'].value_counts()

In [ ]:
rhp_train_df['PULSOS'].value_counts()

In [ ]:
rhp_train_df['B2'].value_counts()

In [ ]:
rhp_train_df['MOTIVO1'].value_counts()

In [ ]:
rhp_train_df['MOTIVO2'].value_counts()

In [ ]:
rhp_train_df['PPA'].value_counts()

In [ ]:
rhp_train_df['CLASSE'].value_counts()

---
### Pré-processamento

Nesta seção, as funções da etapa de pré-processamento dos dados devem ser implementadas e aplicadas (se necessário).

In [46]:
rhp_train_preproc_df = preprocess(rhp_train_df)

In [ ]:
rhp_train_preproc_df.shape

In [ ]:
rhp_train_preproc_df.isna().sum()

---
### Experimento

Nesta seção, o experimento deve ser conduzido, utilizando os protocolos experimentais padrões e testando diferentes modelos.

In [49]:
X = rhp_train_preproc_df.drop(columns=['id', 'classe']).values
y = rhp_train_preproc_df['classe']

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y)

In [ ]:
print(y_train.value_counts())
print(y_test.value_counts())

In [52]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "chebyshev"],
}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, verbose=1)
model_train_knn = ModelTrain(grid_knn)
model_train_knn.train(X_train_scaled, y_train)

In [ ]:
param_grid_nb = {"var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}
grid_nb = GridSearchCV(GaussianNB(), param_grid_nb, cv=5, verbose=1)
model_train_nb = ModelTrain(grid_nb)
model_train_nb.train(X_train_scaled, y_train)

In [ ]:
param_grid_mlp = {
    "alpha": [0.1, 0.01, 0.0001],
    "hidden_layer_sizes": [(100,), (50,), (25,), (10,)],
    "solver": ["adam", "sgd"],
    "activation": ["relu", "tanh"],
}
grid_mlp = GridSearchCV(MLPClassifier(max_iter=1000), param_grid_mlp, cv=5, verbose=1)
model_train_mlp = ModelTrain(grid_mlp)
model_train_mlp.train(X_train_scaled, y_train)

In [ ]:
param_grid_svc = {"C": [0.1, 1, 10], "kernel": ["rbf", "poly", "sigmoid"]}
grid_svc = GridSearchCV(SVC(probability=True), param_grid_svc, cv=5, verbose=1)
model_train_svc = ModelTrain(grid_svc)
model_train_svc.train(X_train_scaled, y_train)

In [ ]:
param_grid_lr = {"C": [0.1, 1, 10, 100], "tol": [1e-4, 1e-3, 1e-2]}
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000), param_grid_lr, cv=5, verbose=1
)
model_train_lr = ModelTrain(grid_lr)
model_train_lr.train(X_train_scaled, y_train)

---
### Análise dos Resultados

Nesta seção, os resultados devem ser exibidos através de tabelas e gráficos, comparados e profundamente analisados.

In [58]:
X_test_scaled = scaler.transform(X_test)

In [ ]:
evaluate(model_train_knn.get_model(), X_test_scaled, y_test, model_name="KNN")

In [ ]:
evaluate(model_train_lr.get_model(), X_test_scaled, y_test, model_name="LR")

In [ ]:
evaluate(model_train_nb.get_model(), X_test_scaled, y_test, model_name="NB")

In [ ]:
evaluate(model_train_svc.get_model(), X_test_scaled, y_test, model_name="SVC")

In [ ]:
evaluate(model_train_mlp.get_model(), X_test_scaled, y_test, model_name="MLP")

---
### Competição

Nesta seção, os modelos serão treinados com toda a base de treino e a base de teste será as observações não rotuladas

In [64]:
scaler_competition = StandardScaler()
X_train_scaled_comp = scaler_competition.fit_transform(X)

In [ ]:
model_train_knn.train(X_train_scaled_comp, y)

In [ ]:
model_train_nb.train(X_train_scaled_comp, y)

In [ ]:
model_train_mlp.train(X_train_scaled_comp, y)

In [ ]:
model_train_svc.train(X_train_scaled_comp, y)

In [ ]:
model_train_lr.train(X_train_scaled_comp, y)

In [70]:
rhp_test_preproc_df = preprocess(rhp_test_df, test=True)
X_test_comp = rhp_test_preproc_df.drop(columns=["id"])
X_test_scaled_comp = scaler_competition.transform(X_test_comp.values)

In [71]:
for model_train in [
    model_train_knn,
    model_train_nb,
    model_train_mlp,
    model_train_svc,
    model_train_lr,
]:
    model = model_train.get_model()
    y_pred = model.predict_proba(X_test_scaled_comp)
    y_pred_proba_df = pd.DataFrame(
        {"Id": rhp_test_preproc_df["id"], "Predicted": y_pred[:, 1]}
    )
    if y_pred_proba_df.shape != (3146, 2):
        raise ValueError(
            "Shape of the prediction DataFrame is incorrect. It should be (3146, 2)"
        )
    y_pred_proba_df.to_csv(
        "data/{}.csv".format(model_train.get_model_name()), index=False
    )